# SimVPv2 Training Notebook — PM2.5 Forecasting (Kavin Folder)

This notebook trains a SimVPv2-style spatiotemporal model using Kavin `src` code and Kavin configs. The model consumes `(B, T_in, C, H, W)` and predicts all 16 future PM2.5 frames in one forward pass.

In [ ]:
# Section 1: Kaggle Runtime Bootstrap (Kavin-Only Paths)
import os, sys

KAGGLE_DATA_ROOT = "/kaggle/input/datasets/khushisingh942004/aisehack"
KAVIN_DIR = "/kaggle/input/datasets/sasidhar412/kavin-pm25-src/ANRF_AISEHack_Code/Kavin"
DATA_DIR = KAGGLE_DATA_ROOT
CKPT_DIR = "/kaggle/temp"
OUT_DIR = "/kaggle/working"

if os.path.exists('/kaggle'):
    os.environ['AISEHACK_DATA'] = DATA_DIR
    SRC_ROOT = KAVIN_DIR
else:
    SRC_ROOT = os.path.abspath('../Kavin')
    DATA_DIR = os.path.abspath('../aisehack-theme-2')
    CKPT_DIR = os.path.abspath('./temp')
    OUT_DIR = os.path.abspath('./working')

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"AISEHACK_DATA: {os.environ.get('AISEHACK_DATA', 'not set')}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"CKPT_DIR: {CKPT_DIR}")
print(f"OUT_DIR: {OUT_DIR}")
assert 'Kavin' in SRC_ROOT, "All imports/paths must resolve under Kavin folder!"

In [ ]:
# Section 2: Repository Import, Seed, and Config Load
import random
import numpy as np
import torch
from src.config import load_config
from src.utils import seed_everything, print_device_info

cfg = load_config()
seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

# Ensure data path is correct for loader
cfg['paths']['data'] = DATA_DIR

print(f"Batch size: {cfg['training']['batch_size_train']}")
print(f"Learning rate: {cfg['training']['lr']}")
print(f"Scheduler: {cfg['training'].get('scheduler', 'coswr')} (T0={cfg['training'].get('t0_epochs', 10)} epochs, T_mult={cfg['training'].get('t_mult', 2)})")
print(f"Epochs: {cfg['training']['epochs']}  |  Patience: {cfg['training']['patience']}")
print(f"Forecast horizon: {cfg['time']['t_out']}")
print(f"lambda_d: {cfg['training']['lambda_d']}")
print(f"lambda_p: {cfg['training']['lambda_p']}")


In [ ]:
# Section 3: Load ALL months and stack base normalized tensor for SimVP
from src.data import load_all_months, load_minmax_bounds
import numpy as np
import gc

def stack_months(month_dicts, feature_list):
    """Stack list of month dicts into (N, C, T, H, W) float32 tensor."""
    T_min = min(d[feature_list[0]].shape[0] for d in month_dicts)
    return np.stack([
        np.stack([d[feat][:T_min] for feat in feature_list], axis=0)
        for d in month_dicts
    ], axis=0).astype(np.float32), T_min

bounds = load_minmax_bounds(cfg)
all_months = cfg['data']['months']
print('Using months:', all_months)
months_data = load_all_months(cfg, all_months, bounds)
base_feature_list = list(cfg['features']['base'])
tensor, T_min = stack_months(months_data, base_feature_list)

del months_data
gc.collect()

print(f"Base tensor shape (float32): {tensor.shape}  ({tensor.nbytes / 1e9:.2f} GB)")
print(f"T_min across months: {T_min}")
print(f"Base channels: {len(base_feature_list)} -> {base_feature_list}")

In [ ]:
# Section 4: (no-op) Lags and lat/lon are already built inside add_physical_features.
# Keeping this cell to avoid re-numbering but NOT allocating any extra arrays.
print("Engineered channels already in tensor — skipping duplicate computation.")
import gc; gc.collect()


In [ ]:
# Section 6: Build SimVPv2 model
from src.model import build_model

cfg['model']['type'] = 'simvpv2'
cfg['model']['simvp_in_channels'] = 24  # cpm25 + met + engineered + emis + time + priors
model = build_model(cfg)

print(f"Model type: {cfg['model']['type']}")
print(f"SimVP input channels: {cfg['model']['simvp_in_channels']}")
print(f"T_in: {cfg['time']['t_in']} | T_out: {cfg['time']['t_out']}")

In [ ]:
# Section 7: Model sanity info
print('Model type:', cfg['model']['type'])
print('SimVP hidden:', cfg['model'].get('simvp_hidden'))
print('Temporal layers:', cfg['model'].get('simvp_temporal_layers'))

In [ ]:
# Section 8: Physics-loss config info
print('lambda_d:', cfg['training'].get('lambda_d'))
print('lambda_p target:', cfg['training'].get('lambda_p'))
print('physics_warmup_epochs:', cfg['training'].get('physics_warmup_epochs', 0))

In [ ]:
# Section 9: Effective input spec
print('SimVP input layout: (B, T, C, H, W)')
print('Configured T_in:', cfg['time']['t_in'])
print('Configured output horizon:', cfg['time']['t_out'])

In [ ]:
# Section 10: Training hyperparameters
print(f"epochs: {cfg['training']['epochs']}")
print(f"lr: {cfg['training']['lr']}")
print(f"scheduler: {cfg['training'].get('scheduler')}")
print(f"batch_size_train: {cfg['training']['batch_size_train']}")

In [ ]:
# Section 10b: Create SimVP DataLoaders (with temporal firewall split)
from src.data import make_simvp_dataloaders

train_dl, val_dl = make_simvp_dataloaders(
    cfg,
    tensor=tensor,
    feature_names=base_feature_list,
    month_names=all_months,
 )
print('SimVP DataLoaders created for training and validation.')

In [ ]:
print("tensor.shape:", tensor.shape)

In [ ]:
# Optional diagnostics (do not mutate tensor before DataLoader creation)
print("Current tensor shape:", tensor.shape)
print("Expected training window (t_in):", cfg['time'].get('t_in', 16))
print("Expected forecast horizon (t_out):", cfg['time'].get('t_out', 16))

In [ ]:
# Section 11: Train Loop Execution with Logging
from src.train import train

xb_dbg, yb_dbg = next(iter(train_dl))
print('Debug xb shape:', xb_dbg.shape)  # expected (B, T_in, C, H, W)
print('Debug yb shape:', yb_dbg.shape)  # expected (B, T_out, 1, H, W)
assert xb_dbg.ndim == 5, f"Expected xb to be (B,T,C,H,W), got {xb_dbg.shape}"
assert yb_dbg.ndim == 5, f"Expected yb to be (B,T_out,1,H,W), got {yb_dbg.shape}"
assert xb_dbg.shape[1] == cfg['time']['t_in'], f"Expected T_in={cfg['time']['t_in']}, got {xb_dbg.shape[1]}"
assert yb_dbg.shape[1] == cfg['time']['t_out'], f"Expected T_out={cfg['time']['t_out']}, got {yb_dbg.shape[1]}"

history = train(cfg, model, train_dl, val_dl, bounds=bounds)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.plot(history['train_loss'], label='Train RMSE (norm)', alpha=0.8)
ax.plot(history['val_loss'],   label='Val RMSE (norm)',   alpha=0.9)
if 'val_persistence' in history:
    ax.plot(history['val_persistence'], label='Val persistence RMSE', alpha=0.8)
ax.set_xlabel('Epoch');  ax.set_ylabel('Normalized RMSE')
ax.set_title('Training Curves — Normalized Space')
ax.legend();  ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Section 12: Notebook Cell: Checkpoint Save/Load Compatible with exp_01_baseline.ipynb
import torch

# Save model checkpoint
torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'best_model.pt'))
print('Model checkpoint saved.')

# Load and sanity check
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, 'best_model.pt'), map_location=cfg['device']))
print('Model checkpoint loaded and verified.')

In [ ]:
# Section 13: Checkpoint save/load
import os
import torch

os.makedirs(CKPT_DIR, exist_ok=True)
ckpt_path = os.path.join(CKPT_DIR, 'best_model.pt')
torch.save(model.state_dict(), ckpt_path)
print('Model checkpoint saved to:', ckpt_path)

state = torch.load(ckpt_path, map_location=cfg['device'])
model.load_state_dict(state)
model.eval()
print('Checkpoint loaded and model set to eval mode.')

In [ ]:
# Section 14: Inference moved to standalone notebook
print('Use Kavin/notebooks/inference_standalone.ipynb for submission inference.')
print('This training notebook now focuses on SimVP training only.')

In [ ]:
# Section 14: Reliable prediction export with verification (atomic save)
import os
import time
import numpy as np

os.makedirs(OUT_DIR, exist_ok=True)

if 'preds' not in globals() or preds is None:
    raise RuntimeError('`preds` is missing. Run Cell 13 first.')

preds = np.asarray(preds, dtype=np.float32)
if not np.isfinite(preds).all():
    raise RuntimeError('`preds` contains NaN/Inf; aborting save.')

final_path = os.path.join(OUT_DIR, 'preds.npy')
tmp_path = final_path + '.tmp'
backup_path = os.path.join(OUT_DIR, f"preds_backup_{int(time.time())}.npy")

# Save temp -> fsync -> atomic replace
with open(tmp_path, 'wb') as f:
    np.save(f, preds)
    f.flush()
    os.fsync(f.fileno())
os.replace(tmp_path, final_path)

# Optional backup copy for extra safety
np.save(backup_path, preds)

# Read-back verification
loaded = np.load(final_path, mmap_mode='r')
assert loaded.shape == preds.shape, f"Saved shape mismatch: {loaded.shape} vs {preds.shape}"

print(f'Predictions saved to {final_path}')
print(f'Backup copy saved to {backup_path}')
print(f'Final file size: {os.path.getsize(final_path) / (1024**2):.2f} MB')